# Remaining RG3 gas misses: cooling / pressure-fall signature
Analyze all12 missed by47 mean anchor. Predeclared auxiliary signature: barrel5 current delta<0 and pressure current delta<=0.
Train cold-signature gas vs training normals; exclude other training gas rather than relabel it normal. Query labels never used to define training subset.
Six current-shot features. RF+three-C logistic ensemble; gated causal score and inner-normal0.5/1% thresholds. Fixed guarded mean branch requires train-inner TP gain and <=2 added FP.
Guard selection uses original RF inner scores, NOT global47 OOF predictions that could contain query labels. Outer anchor kept frozen; no prediction deletion.
Signature ideation is test-informed exploratory work, not independent verification.

In [ ]:
from pathlib import Path
import json
import numpy as np,pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
nb=json.loads((root/'notebooks/22_rg3_nested_signal_ensemble.ipynb').read_text(encoding='utf-8'))
exec(''.join(nb['cells'][1]['source']).split('scores=[];thresholds=[];foldlogs=[]')[0])
anchor=pd.read_csv(tables/'rg3_innovation_seed_sensitivity_predictions.csv',parse_dates=['TimeStamp'])
anchor=anchor.loc[anchor.model.eq('three_seed_mean'),['scheme','TimeStamp','prediction','actual']].rename(columns={'prediction':'anchor_prediction'})
oldinner=pd.read_csv(tables/'rg3_rank_fusion_inner_scores.csv',parse_dates=['TimeStamp'])
dp=d.Max_Injection_Pressure__delta1.to_numpy(float)
dt=d.Barrel_Temperature_5__delta1.to_numpy(float)
cold=(dt<0)&(dp<=0)
F=np.column_stack([dt,dp,d.Hopper_Temperature__delta1.to_numpy(float),d.Cushion_Position__delta1.to_numpy(float),(d.Barrel_Temperature_2__delta1-d.Barrel_Temperature_5__delta1).to_numpy(float),dt*dp])
feature_names=['barrel5_delta','pressure_delta','hopper_delta','cushion_delta','barrel2_minus_barrel5_delta','thermal_pressure_delta_product']
pd.DataFrame(F,columns=feature_names).assign(TimeStamp=d.TimeStamp).to_csv(tables/'rg3_cold_signature_features.csv',index=False,encoding='utf-8-sig')
miss=d[['TimeStamp','comparison_block']].merge(anchor.loc[anchor.scheme.eq('leave_one_block_out')],on='TimeStamp',validate='one_to_one')
miss['barrel5_delta']=dt;miss['pressure_delta']=dp;miss['cold_signature']=cold
miss['history_missing']=d[stdcols].isna().any(axis=1).to_numpy()
miss.loc[miss.actual.eq(1)&miss.anchor_prediction.eq(0)].to_csv(tables/'rg3_remaining12_signature_audit.csv',index=False,encoding='utf-8-sig')
memo={}
def predict(ref,q):
    key=(tuple(ref),tuple(q))
    if key in memo:return memo[key]
    assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
    chosen=ref[(y[ref]==0)|((y[ref]==1)&cold[ref])]
    if len(np.unique(y[chosen]))<2:return np.zeros((len(q),4))
    imp=SimpleImputer(strategy='median',keep_empty_features=True)
    tr=imp.fit_transform(F[chosen]);te=imp.transform(F[q])
    rf=RandomForestClassifier(n_estimators=100,max_depth=5,min_samples_leaf=3,class_weight='balanced_subsample',random_state=42,n_jobs=2)
    a=rf.fit(tr,y[chosen]).predict_proba(te)[:,1]
    sc=RobustScaler().fit(tr[y[chosen]==0]);tr=sc.transform(tr);te=sc.transform(te)
    linear=[]
    for C in [.1,1.,10.]:
        m=LogisticRegression(C=C,class_weight='balanced',max_iter=2000,random_state=42)
        linear.append(m.fit(tr,y[chosen]).predict_proba(te)[:,1])
    b=np.mean(linear,axis=0)
    result=np.column_stack([a,b,(a+b)/2,np.sqrt(a*b)])
    # The cold signature is a predeclared auxiliary target, not a new defect label.
    result[~cold[q]]=0.
    memo[key]=result
    return result
models=['cold_RF','cold_logit3','cold_mean','cold_geometric']
records=[];choices=[];folds=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,b in enumerate(blocks):
        q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        if len(np.unique(y[ref]))<2:continue
        outer=predict(ref,q);inner=np.full((len(ref),4),np.nan)
        for ib in groups.iloc[ref].unique():
            mask=groups.iloc[ref].eq(ib).to_numpy()
            if (~mask).any():inner[mask]=predict(ref[~mask],ref[mask])
        anc=anchor.loc[anchor.scheme.eq(scheme)].set_index('TimeStamp').loc[d.TimeStamp.iloc[q],'anchor_prediction'].to_numpy(bool)
        # Choose new branch using original RF train-inner scores only.
        # Never use global anchor OOF predictions as train calibration, which could leak query labels.
        ic=oldinner.loc[oldinner.scheme.eq(scheme)&oldinner.heldout_block.eq(b)].set_index('TimeStamp').loc[d.TimeStamp.iloc[ref]]
        cut_old=float(np.quantile(ic.loc[y[ref]==0,'raw_baseline_score'],.99,method='higher'))
        oldpr=ic.raw_baseline_score.gt(cut_old).to_numpy()
        tn0,fp0,fn0,tp0=confusion_matrix(y[ref],oldpr,labels=[0,1]).ravel()
        f0=2*tp0/(2*tp0+fp0+fn0) if 2*tp0+fp0+fn0 else 0.
        for k,name in enumerate(models):
            norm=inner[(y[ref]==0)&np.isfinite(inner[:,k]),k]
            for rate in [.005,.01]:
                cut=float(np.quantile(norm,1-rate,method='higher')) if len(norm) else 1.
                new_outer=outer[:,k]>cut
                for mode,pr in [('new_only',new_outer),('OR_anchor',anc|new_outer)]:
                    for l,i in enumerate(q):records.append(dict(scheme=scheme,model=name,budget=rate,combine=mode,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(outer[l,k]),threshold=cut,prediction=int(pr[l]),anchor_prediction=int(anc[l])))
                if name=='cold_mean' and rate==.005:
                    valid=np.isfinite(inner[:,k]);new_inner=inner[:,k]>cut
                    tn,fp,fn,tp=confusion_matrix(y[ref][valid],(oldpr|new_inner)[valid],labels=[0,1]).ravel()
                    f=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
                    allow=bool(tp>tp0 and fp-fp0<=2 and f>f0)
                    pr=anc|(new_outer&allow)
                    choices.append(dict(scheme=scheme,block=b,add_allowed=allow,inner_added_TP=int(tp-tp0),inner_added_FP=int(fp-fp0),inner_F1=f,original_inner_F1=f0))
                    for l,i in enumerate(q):records.append(dict(scheme=scheme,model='cold_mean_guarded',budget=rate,combine='OR_anchor',TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(outer[l,k]),threshold=cut,prediction=int(pr[l]),anchor_prediction=int(anc[l])))
        folds.append(dict(scheme=scheme,block=b,train=len(ref),train_gas=int(y[ref].sum()),train_cold_gas=int(((y[ref]==1)&cold[ref]).sum()),test=len(q),test_gas=int(y[q].sum())))
        print('Finished cold-signature branch',scheme,b,flush=True)
pp=pd.DataFrame(records);rows=[]
for (scheme,name,rate,mode),g in pp.groupby(['scheme','model','budget','combine'],sort=False):
    tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
    rows.append(dict(scheme=scheme,model=name,budget=rate,combine=mode,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.))
metrics=pd.DataFrame(rows)
pp.to_csv(tables/'rg3_cold_signature_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(tables/'rg3_cold_signature_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(choices).to_csv(tables/'rg3_cold_signature_inner_choices.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(folds).to_csv(tables/'rg3_cold_signature_fold_audit.csv',index=False,encoding='utf-8-sig')
missed=miss.loc[miss.actual.eq(1)&miss.anchor_prediction.eq(0)]
normal=miss.loc[miss.actual.eq(0)]
print('Remaining misses',len(missed),'barrel5 falling',int(missed.barrel5_delta.lt(0).sum()),'cold signature',int(missed.cold_signature.sum()),'history missing',int(missed.history_missing.sum()))
print('Normal cold signature',int(normal.cold_signature.sum()),'/',len(normal))
print(metrics.sort_values(['scheme','F1','FP'],ascending=[True,False,True]).groupby('scheme').head(6).to_string(index=False))
print(metrics.loc[metrics.model.eq('cold_mean_guarded')].to_string(index=False))


## Executed results
Remaining12: barrel5 falling7, cold-pressure signature5, history missing1. Normals with cold signature150/566. Logistic auxiliary adds01:04:15 but whole6/12/F1=0.343, worse than anchor5/6/F1=0.357. Guarded branch keeps anchor5/6 and forward3/4; not adopted.
See reports/rg3_current_innovation_confirmation.md and reports/rg3_remaining12_cold_signature_review.md.